In [48]:
%pip install -U langchain langchain-ollama requests

Defaulting to user installation because normal site-packages is not writeable
     |████████████████████████████████| 1.0 MB 666 kB/s eta 0:00:01
     |████████████████████████████████| 461 kB 643 kB/s eta 0:00:01
     |████████████████████████████████| 472 kB 2.2 MB/s eta 0:00:01
     |████████████████████████████████| 396 kB 38 kB/s eta 0:00:011
     |████████████████████████████████| 2.2 MB 2.6 MB/s eta 0:00:01
     |████████████████████████████████| 66 kB 3.3 MB/s eta 0:00:01
     |████████████████████████████████| 572 kB 2.4 MB/s eta 0:00:01
     |████████████████████████████████| 245 kB 3.2 MB/s eta 0:00:01
     |████████████████████████████████| 640 kB 2.9 MB/s eta 0:00:01
     |████████████████████████████████| 54 kB 2.2 MB/s eta 0:00:01
     |████████████████████████████████| 1.9 MB 2.9 MB/s eta 0:00:01
  Attempting uninstall: packaging
    Found existing installation: packaging 26.3
    Uninstalling packaging-26.3:
      Successfully uninstalled packaging-26.3
You should cons

In [33]:
import requests

GRAPHDB_ENDPOINT = (
    "http://localhost:7200/repositories/metagenomicKG"
)

session = requests.Session()

The history saving thread hit an unexpected error (OperationalError('attempt to write a readonly database')).History will not be written to the database.


In [34]:
import requests
import re

GRAPHDB_ENDPOINT = "http://localhost:7200/repositories/metagenomicKG"

def run_sparql(query, timeout=120):
    """Execute a read-only SPARQL SELECT query against GraphDB."""

    # Remove PREFIX and BASE declarations for validation.
    cleaned_query = re.sub(
        r"(?im)^[ \t]*(?:PREFIX[ \t]+\w*:[ \t]*<[^>]+>|BASE[ \t]+<[^>]+>)[ \t]*",
        "",
        query
    ).lstrip()

    # Allow SELECT queries only.
    if not re.match(r"(?i)^SELECT\b", cleaned_query):
        raise ValueError(
            f"Only SELECT queries are permitted. "
            f"Query starts with: {cleaned_query[:100]!r}"
        )

    response = requests.get(
        GRAPHDB_ENDPOINT,
        params={"query": query},
        headers={"Accept": "application/sparql-results+json"},
        timeout=timeout
    )

    response.raise_for_status()
    data = response.json()

    return [
        {key: value["value"] for key, value in row.items()}
        for row in data["results"]["bindings"]
    ]

In [35]:
query = """
SELECT ?s ?p ?o
WHERE {
    ?s ?p ?o .
}
LIMIT 5
"""

results = run_sparql(query)

for row in results:
    print(row)
    

{'s': 'http://www.w3.org/1999/02/22-rdf-syntax-ns#type', 'p': 'http://www.w3.org/1999/02/22-rdf-syntax-ns#type', 'o': 'http://www.w3.org/1999/02/22-rdf-syntax-ns#Property'}
{'s': 'http://www.w3.org/2000/01/rdf-schema#subPropertyOf', 'p': 'http://www.w3.org/1999/02/22-rdf-syntax-ns#type', 'o': 'http://www.w3.org/1999/02/22-rdf-syntax-ns#Property'}
{'s': 'http://www.w3.org/2000/01/rdf-schema#subPropertyOf', 'p': 'http://www.w3.org/1999/02/22-rdf-syntax-ns#type', 'o': 'http://www.w3.org/2002/07/owl#TransitiveProperty'}
{'s': 'http://www.w3.org/2000/01/rdf-schema#subClassOf', 'p': 'http://www.w3.org/1999/02/22-rdf-syntax-ns#type', 'o': 'http://www.w3.org/1999/02/22-rdf-syntax-ns#Property'}
{'s': 'http://www.w3.org/2000/01/rdf-schema#subClassOf', 'p': 'http://www.w3.org/1999/02/22-rdf-syntax-ns#type', 'o': 'http://www.w3.org/2002/07/owl#TransitiveProperty'}


In [60]:
def find_entities(name, limit=10):
    escaped_name = name.replace("\\", "\\\\").replace('"', '\\"')

    query = f"""
    PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
    PREFIX skos: <http://www.w3.org/2004/02/skos/core#>

    SELECT DISTINCT ?entity ?label
    WHERE {{
        {{
            ?entity rdfs:label ?label .
        }}
        UNION
        {{
            ?entity skos:altLabel ?label .
        }}

        FILTER(
            LCASE(STR(?label)) = LCASE("{escaped_name}")
        )
    }}
    LIMIT {int(limit)}
    """

    return run_sparql(query)

In [64]:
find_entities("Escherichia coli")

[{'entity': 'https://w3id.org/metagenomickg/Microbe:480684',
  'label': 'Escherichia coli'},
 {'entity': 'https://w3id.org/metagenomickg/Microbe:480685',
  'label': 'Escherichia coli'},
 {'entity': 'https://w3id.org/metagenomickg/Microbe:480686',
  'label': 'Escherichia coli'},
 {'entity': 'https://w3id.org/metagenomickg/Microbe:480687',
  'label': 'Escherichia coli'},
 {'entity': 'https://w3id.org/metagenomickg/Microbe:480688',
  'label': 'Escherichia coli'},
 {'entity': 'https://w3id.org/metagenomickg/Microbe:480689',
  'label': 'Escherichia coli'},
 {'entity': 'https://w3id.org/metagenomickg/Microbe:480690',
  'label': 'Escherichia coli'},
 {'entity': 'https://w3id.org/metagenomickg/Microbe:480691',
  'label': 'Escherichia coli'},
 {'entity': 'https://w3id.org/metagenomickg/Microbe:480692',
  'label': 'Escherichia coli'},
 {'entity': 'https://w3id.org/metagenomickg/Microbe:480693',
  'label': 'Escherichia coli'}]

In [62]:
def inspect_entity(entity_uri):
    query = f"""
    PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
    PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
    PREFIX dcterms: <http://purl.org/dc/terms/>

    SELECT ?property ?value
    WHERE {{
        <{entity_uri}> ?property ?value .
        FILTER(
            ?property = rdf:type ||
            ?property = rdfs:label ||
            ?property = rdfs:comment ||
            ?property = dcterms:source
        )
    }}
    LIMIT 30
    """
    return run_sparql(query)

In [63]:
inspect_entity(
    "https://w3id.org/metagenomickg/Microbe:480684"
)

[{'property': 'http://www.w3.org/1999/02/22-rdf-syntax-ns#type',
  'value': 'https://w3id.org/biolink/vocab/OrganismTaxon'},
 {'property': 'http://www.w3.org/2000/01/rdf-schema#label',
  'value': 'Escherichia coli'},
 {'property': 'http://www.w3.org/2000/01/rdf-schema#comment',
  'value': "[('rank', 'species'), ('MicroPhenoDB Description', 'A common, gram negative gut bacterium that has been studied intensively by geneticists because of its small genome size, normal lack of pathogenicity, and ease of growth in the laboratory.'), ('taxid', '562')]"},
 {'property': 'http://purl.org/dc/terms/source', 'value': 'MicroPhenoDB'},
 {'property': 'http://purl.org/dc/terms/source', 'value': 'GTDB'}]

In [40]:
def get_relationships(entity_uri, limit=20):
    limit = max(1, min(int(limit), 100))

    query = f"""
    SELECT ?direction ?predicate ?related_entity
    WHERE {{
        {{
            <{entity_uri}> ?predicate ?related_entity .
            BIND("outgoing" AS ?direction)
        }}
        UNION
        {{
            ?related_entity ?predicate <{entity_uri}> .
            BIND("incoming" AS ?direction)
        }}
    }}
    LIMIT {limit}
    """
    return run_sparql(query)

In [41]:
get_relationships(
    "https://w3id.org/metagenomickg/Microbe:480684"
)

[{'direction': 'outgoing',
  'predicate': 'http://www.w3.org/1999/02/22-rdf-syntax-ns#type',
  'related_entity': 'https://w3id.org/biolink/vocab/OrganismTaxon'},
 {'direction': 'outgoing',
  'predicate': 'http://www.w3.org/2000/01/rdf-schema#comment',
  'related_entity': "[('rank', 'species'), ('MicroPhenoDB Description', 'A common, gram negative gut bacterium that has been studied intensively by geneticists because of its small genome size, normal lack of pathogenicity, and ease of growth in the laboratory.'), ('taxid', '562')]"},
 {'direction': 'outgoing',
  'predicate': 'http://www.w3.org/2000/01/rdf-schema#label',
  'related_entity': 'Escherichia coli'},
 {'direction': 'outgoing',
  'predicate': 'https://w3id.org/biolink/vocab/superclass_of',
  'related_entity': 'https://w3id.org/metagenomickg/Microbe:480685'},
 {'direction': 'outgoing',
  'predicate': 'https://w3id.org/biolink/vocab/superclass_of',
  'related_entity': 'https://w3id.org/metagenomickg/Microbe:480686'},
 {'direction'

In [42]:
def search_entities_detailed(name, limit=10):
    limit = max(1, min(int(limit), 100))

    escaped_name = (
        name.replace("\\", "\\\\")
            .replace('"', '\\"')
    )

    query = f"""
    PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
    PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
    PREFIX skos: <http://www.w3.org/2004/02/skos/core#>

    SELECT DISTINCT ?entity ?label ?type ?description
    WHERE {{
        {{
            ?entity rdfs:label ?label .
        }}
        UNION
        {{
            ?entity skos:altLabel ?label .
        }}

        OPTIONAL {{
            ?entity rdf:type ?type .
        }}

        OPTIONAL {{
            ?entity rdfs:comment ?description .
        }}

        FILTER(
            LCASE(STR(?label)) = LCASE("{escaped_name}")
        )
    }}
    LIMIT {limit}
    """

    return run_sparql(query)

In [65]:
search_entities_detailed("Bifidobacterium longum")

[{'entity': 'https://w3id.org/metagenomickg/Microbe:698667',
  'label': 'Bifidobacterium longum',
  'type': 'https://w3id.org/biolink/vocab/OrganismTaxon',
  'description': "[('rank', 'species'), ('MicroPhenoDB Description', 'A species of anaerobic, Gram-positive, rod shaped bacteria assigned to the phylum Actinobacteria. This species is nonmotile, nonsporeforming, catalase and indole negative and ferments arabinose, xylose, ribose, lactose and melezitose. B. longum is non-pathogenic, and it helps maintain a healthy digestive system.'), ('taxid', '216816')]"},
 {'entity': 'https://w3id.org/metagenomickg/Microbe:698668',
  'label': 'Bifidobacterium longum',
  'type': 'https://w3id.org/biolink/vocab/OrganismTaxon',
  'description': "[('taxid', '216816'), ('rank', 'strain')]"},
 {'entity': 'https://w3id.org/metagenomickg/Microbe:698669',
  'label': 'Bifidobacterium longum',
  'type': 'https://w3id.org/biolink/vocab/OrganismTaxon',
  'description': "[('taxid', '216816'), ('rank', 'strain')

In [44]:
def get_biological_relationships(entity_uri, limit=50):
    limit = max(1, min(int(limit), 200))

    query = f"""
    PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
    PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
    PREFIX biolink: <https://w3id.org/biolink/vocab/>

    SELECT ?direction ?predicate ?related_entity ?related_label
    WHERE {{
        {{
            <{entity_uri}> ?predicate ?related_entity .
            BIND("outgoing" AS ?direction)
        }}
        UNION
        {{
            ?related_entity ?predicate <{entity_uri}> .
            BIND("incoming" AS ?direction)
        }}

        FILTER(
            ?predicate != rdf:type &&
            ?predicate != rdfs:label &&
            ?predicate != rdfs:comment
        )

        OPTIONAL {{
            ?related_entity rdfs:label ?related_label .
        }}
    }}
    LIMIT {limit}
    """

    return run_sparql(query)

In [45]:
get_biological_relationships(
    "https://w3id.org/metagenomickg/Microbe:698668"
)

[{'direction': 'outgoing',
  'predicate': 'https://w3id.org/biolink/vocab/subclass_of',
  'related_entity': 'https://w3id.org/metagenomickg/Microbe:698667',
  'related_label': 'Bifidobacterium longum'},
 {'direction': 'outgoing',
  'predicate': 'https://w3id.org/biolink/vocab/associated_with',
  'related_entity': 'https://w3id.org/metagenomickg/AMR:1732',
  'related_label': '23S rRNA (adenine(2058)-N(6))-methyltransferase Erm(X)'},
 {'direction': 'outgoing',
  'predicate': 'http://purl.org/dc/terms/source',
  'related_entity': 'GTDB'},
 {'direction': 'outgoing',
  'predicate': 'http://www.w3.org/2004/02/skos/core#altLabel',
  'related_entity': 'GTDB:GCF_003468825.1'},
 {'direction': 'outgoing',
  'predicate': 'https://w3id.org/metagenomickg/vocab/link',
  'related_entity': 'https://www.ncbi.nlm.nih.gov/Taxonomy/Browser/wwwtax.cgi?mode=Info&id=216816'},
 {'direction': 'outgoing',
  'predicate': 'https://w3id.org/metagenomickg/vocab/link',
  'related_entity': 'https://gtdb.ecogenomic.org

In [46]:
query = """
PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
PREFIX biolink: <https://w3id.org/biolink/vocab/>

SELECT ?predicate (COUNT(*) AS ?number_of_relationships)
WHERE {
    ?microbe rdf:type biolink:OrganismTaxon .
    ?disease rdf:type biolink:Disease .

    ?microbe ?predicate ?disease .

    FILTER(?microbe != ?disease)
}
GROUP BY ?predicate
ORDER BY DESC(?number_of_relationships)
LIMIT 30
"""

run_sparql(query)

[{'predicate': 'https://w3id.org/biolink/vocab/associated_with',
  'number_of_relationships': '12453'}]

In [47]:
predicate_categories = {
    "taxonomy_hierarchy": [
        "subclass_of",
        "superclass_of",
        "same_as",
        "close_match"
    ],

    "genetic_biological_associations": [
        "genetically_associated_with",
        "associated_with",
        "related_to",
        "correlated_with",
        "causes",
        "affects",
        "contributes_to",
        "regulates",
        "temporally_related_to"
    ],

    "molecular_interactions_functions": [
        "interacts_with",
        "physically_interacts_with",
        "chemically_similar_to",
        "catalyzes",
        "gene_product_of",
        "is_sequence_variant_of",
        "has_input",
        "produces",
        "has_metabolite",
        "disrupts"
    ],

    "disease_phenotype_treatment": [
        "has_phenotype",
        "biomarker_for",
        "manifestation_of",
        "treats",
        "applied_to_treat",
        "contraindicated_in",
        "disease_has_basis_in",
        "drug_regulatory_status_world_wide"
    ],

    "processes_composition_location": [
        "participates_in",
        "has_participant",
        "has_part",
        "has_member",
        "located_in"
    ],

    "metadata_provenance": [
        "label",
        "comment",
        "altLabel",
        "link",
        "source",
        "type",
        "is_pathogen"
    ]
}

In [51]:
from langchain_ollama import ChatOllama

llm = ChatOllama(
    model="qwen2.5:7b",
    temperature=0
)
response = llm.invoke(
    "Explain what SPARQL is in two sentences."
)

print(response.content)

SPARQL is a query language for retrieving and manipulating data stored in RDF format, similar to SQL for relational databases. It enables users to ask questions and perform operations on data linked by relationships in a flexible and powerful way.


In [52]:
results = run_sparql("""
SELECT (COUNT(*) AS ?total)
WHERE {
    ?s ?p ?o .
}
""")

print(results)

[{'total': '73330741'}]


In [54]:
from langchain_core.tools import tool
import json

@tool
def find_entities(name: str) -> str:
    """
    Find biological entities in MetagenomicKG by name.
    Use this when the user mentions a microbe, gene, disease,
    metabolite, or other biological entity.
    """
    results = search_entities_detailed(name, limit=10)
    return json.dumps(results, indent=2)
@tool
def get_relationships(entity_uri: str) -> str:
    """
    Retrieve incoming and outgoing biological relationships
    for an entity identified by its full URI.
    """
    results = get_biological_relationships(
        entity_uri,
        limit=30
    )
    return json.dumps(results, indent=2)


@tool
def search_microbes(name: str) -> str:
    """Search MetagenomicKG for a biological entity by its name.
    Input should be a plain entity name, such as Escherichia coli.
    Returns matching entity URIs and labels."""
    results = find_entities(name)

    if not results:
        return f"No entities found for: {name}"

    return "\n".join(
        f"URI: {item['entity']} | Label: {item['label']}"
        for item in results
    )

In [73]:
from langchain.agents import AgentType, initialize_agent

tools = [
    search_microbes,
    get_relationships
]

agent = initialize_agent(
    tools=tools,
    llm=llm,
    agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION,
    verbose=True,
    handle_parsing_errors=True,
    agent_kwargs={
        "prefix": """
You are a biological knowledge graph assistant for MetagenomicKG.

Use the available tools to search for biological entities
and retrieve their relationships.

Rules:
1. Use tools when you need information from the knowledge graph.
2. Use exact entity URIs returned by find_entities.
3. Base answers on retrieved results.
4. Do not invent biological relationships.
5. If the graph lacks sufficient evidence, explain the limitation in 5 sentences only.
"""
    }
)

In [56]:
import langchain

print(langchain.__version__)

0.3.30


In [76]:
response = agent.invoke({
    "input": (
        "Find Bifidobacterium longum in MetagenomicKG "
        "and find relationships other than its subclass or superclass."
    )
})

print(response["output"])



> Entering new AgentExecutor chain...
I need to search for the entity "Bifidobacterium longum" in MetagenomicKG to get its URI and label.
Action: search_microbes
Action Input: Bifidobacterium longum
Observation: URI: https://w3id.org/metagenomickg/Microbe:698667 | Label: Bifidobacterium longum
URI: https://w3id.org/metagenomickg/Microbe:698668 | Label: Bifidobacterium longum
URI: https://w3id.org/metagenomickg/Microbe:698669 | Label: Bifidobacterium longum
URI: https://w3id.org/metagenomickg/Microbe:698670 | Label: Bifidobacterium longum
URI: https://w3id.org/metagenomickg/Microbe:698671 | Label: Bifidobacterium longum
URI: https://w3id.org/metagenomickg/Microbe:698672 | Label: Bifidobacterium longum
URI: https://w3id.org/metagenomickg/Microbe:698673 | Label: Bifidobacterium longum
URI: https://w3id.org/metagenomickg/Microbe:698675 | Label: Bifidobacterium longum
URI: https://w3id.org/metagenomickg/Microbe:698677 | Label: Bifidobacterium longum
URI: https://w3id.org/metagenomickg/Mic

KeyboardInterrupt: 

In [67]:
from langchain_core.tools import tool


In [69]:
print(search_microbes.name)
print(search_microbes.args)
print(search_microbes.invoke({"name": "Escherichia coli"}))

search_microbes
{'name': {'title': 'Name', 'type': 'string'}}
URI: https://w3id.org/metagenomickg/Microbe:480684 | Label: Escherichia coli
URI: https://w3id.org/metagenomickg/Microbe:480685 | Label: Escherichia coli
URI: https://w3id.org/metagenomickg/Microbe:480686 | Label: Escherichia coli
URI: https://w3id.org/metagenomickg/Microbe:480687 | Label: Escherichia coli
URI: https://w3id.org/metagenomickg/Microbe:480688 | Label: Escherichia coli
URI: https://w3id.org/metagenomickg/Microbe:480689 | Label: Escherichia coli
URI: https://w3id.org/metagenomickg/Microbe:480690 | Label: Escherichia coli
URI: https://w3id.org/metagenomickg/Microbe:480691 | Label: Escherichia coli
URI: https://w3id.org/metagenomickg/Microbe:480692 | Label: Escherichia coli
URI: https://w3id.org/metagenomickg/Microbe:480693 | Label: Escherichia coli


In [77]:
print([tool.name for tool in tools])

['search_microbes', 'get_relationships']
